# データの可視化

In [0]:
!pip install japanize-matplotlib

In [0]:
%restart_python

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import japanize_matplotlib


# matplotlib関連の全警告を抑制（フォント警告含む）
warnings.filterwarnings('ignore')

# 日本語表示設定（フォールバックで表示される）
plt.rcParams['axes.unicode_minus'] = False

In [0]:
# データ読み込み
df_sales = spark.read.table("workspace.gold._30_gold_daily_sales_summary")

# Pandas DataFrameに変換（可視化用）
pdf = df_sales.toPandas()

print(f"データ件数: {len(pdf)}件")
print(f"期間: {pdf['sale_date'].min()} 〜 {pdf['sale_date'].max()}")
print(f"店舗数: {pdf['store_name'].nunique()}店舗")
print(f"商品数: {pdf['product_name'].nunique()}商品")

In [0]:
pdf["sale_date"] = pd.to_datetime(pdf["sale_date"])

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("売上分析ダッシュボード", fontsize=18, fontweight="bold")

# 1. 日別売上
daily = pdf.groupby("sale_date")["total_sales_amount"].sum().sort_index()
axes[0, 0].plot(daily.index, daily.values, marker="o")
axes[0, 0].set_title("日別総売上推移")
axes[0, 0].set_ylabel("売上金額（円）")
axes[0, 0].tick_params(axis="x", rotation=45)
axes[0, 0].grid(alpha=0.3)

# 2. 店舗別売上
store = pdf.groupby("store_name")["total_sales_amount"].sum().sort_values(ascending=False)
bars = axes[0, 1].bar(store.index, store.values)
axes[0, 1].set_title("店舗別総売上")
axes[0, 1].tick_params(axis="x", rotation=30)
axes[0, 1].bar_label(bars, labels=[f"{v:,.0f}" for v in store.values], padding=3)

# 3. カテゴリ別構成
category = pdf.groupby("category")["total_sales_amount"].sum()
axes[1, 0].pie(
    category.values,
    labels=category.index,
    autopct="%1.1f%%",
    startangle=90
)
axes[1, 0].set_title("カテゴリ別売上構成")

# 4. 商品別トップ10
product = (
    pdf.groupby("product_name")["total_sales_amount"]
    .sum()
    .nlargest(10)
    .sort_values()
)
bars = axes[1, 1].barh(product.index, product.values)
axes[1, 1].set_title("商品別売上トップ10")
axes[1, 1].set_xlabel("売上金額（円）")
axes[1, 1].bar_label(bars, labels=[f"{v:,.0f}" for v in product.values], padding=3)

plt.tight_layout()
plt.show()